<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook summarizes electronic health record phenotypes for the 1,027 Phase 1 long-read participants. It unions OMOP condition, observation, measurement, drug, and procedure records, maps concepts through SNOMED vocabulary, and produces prevalence bars for selected conditions, per-sample concept counts by OMOP domain, and age-by-sex distributions.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import os
import pandas as pd
from google.cloud import bigquery

In [ ]:
# Long-read sample IDs: one directory per sample in the Controlled Tier long-read BAM location.
!gsutil -u $GOOGLE_PROJECT ls gs://fc-aou-datasets-controlled/pooled/longreads/v7_base/bam/ | xargs -n1 basename > lr_sample_ids.txt

In [ ]:
# Short-read QC metrics table, used here for each participant's sex at birth.
if not os.path.exists("genomic_metrics.tsv"):
    !gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv .
    
sr_metrics = pd.read_table('genomic_metrics.tsv')

In [ ]:
sr_metrics

In [ ]:
# Pull every concept recorded for the long-read participants from five OMOP domains (conditions,
# observations, measurements, drug exposures, procedures) and save the distinct (person, concept, domain) rows.
client = bigquery.Client()
CDR_version = os.getenv("WORKSPACE_CDR")

sample_ids_file = "lr_sample_ids.txt"  # AoU 1027 samples
with open(sample_ids_file, "r") as f:
    sample_ids = [line.strip() for line in f if line.strip()] 
    
sample_ids_str = ",".join(sample_ids)

query = f"""
WITH combined_data AS (
    SELECT person_id, condition_concept_id AS concept_id, 'condition' AS source
    FROM `{CDR_version}.condition_occurrence`
    WHERE person_id IN ({sample_ids_str})

    UNION ALL

    SELECT person_id, observation_concept_id AS concept_id, 'observation' AS source
    FROM `{CDR_version}.observation`
    WHERE person_id IN ({sample_ids_str})

    UNION ALL

    SELECT person_id, measurement_concept_id AS concept_id, 'measurement' AS source
    FROM `{CDR_version}.measurement`
    WHERE person_id IN ({sample_ids_str})

    UNION ALL

    SELECT person_id, drug_concept_id AS concept_id, 'drug_exposure' AS source
    FROM `{CDR_version}.drug_exposure`
    WHERE person_id IN ({sample_ids_str})

    UNION ALL

    SELECT person_id, procedure_concept_id AS concept_id, 'procedure' AS source
    FROM `{CDR_version}.procedure_occurrence`
    WHERE person_id IN ({sample_ids_str})
)
SELECT person_id, concept_id, source FROM combined_data
"""

df_samples = client.query(query).to_dataframe().drop_duplicates()
output_file = "AoU_LR_samples_all_domains.csv"
df_samples.to_csv(output_file, index=False)

In [ ]:
sr_metrics

In [ ]:
!head -n 10 AoU_LR_samples_all_domains.csv

In [ ]:
# Add sex at birth to each row.
# Merge to add sex_at_birth from sr_metrics to df_samples
df_samples = df_samples.merge(
    sr_metrics[["research_id", "sex_at_birth"]],
    left_on="person_id",
    right_on="research_id",
    how="left"
)

# Optionally drop the extra research_id column after merge
df_samples.drop(columns=["research_id"], inplace=True)

# Save to file if needed
df_samples.to_csv("AoU_LR_samples_all_domains_with_sex.csv", index=False)

In [ ]:
# OHDSI vocabulary tables (CONCEPT.csv, CONCEPT_ANCESTOR.csv, ...), stored in the workspace bucket.
!gsutil -u $GOOGLE_PROJECT cp $WORKSPACE_BUCKET/resources/vocabulary.zip .

In [ ]:
!unzip -o vocabulary.zip

In [ ]:
# Attach concept names and classes to each participant's condition concepts.
samples_file = "AoU_LR_samples_all_domains_with_sex.csv"
df_samples = pd.read_csv(samples_file)

df_conditions = df_samples[df_samples["source"] == "condition"][["person_id", "concept_id"]]

vocabulary_file = "CONCEPT.csv"  # from OHDSI
df_vocab = pd.read_csv(vocabulary_file, sep="\t",low_memory=False)

df_vocab = df_vocab[["concept_id", "concept_name", "concept_class_id", "standard_concept"]]
df_merged = pd.merge(df_conditions, df_vocab, on="concept_id", how="left")

output_file = "AoU_condition_with_details.csv"
df_merged.to_csv(output_file, index=False)

In [ ]:
df_merged

In [ ]:
# Map every SNOMED condition concept to all of its ancestors, so that a broad disease term (for example
# 'heart disease') can be matched to every more specific concept beneath it.
import pandas as pd

concept_df = pd.read_csv("CONCEPT.csv", sep='\t',low_memory=False) # from OHDSI
ancestor_df = pd.read_csv("CONCEPT_ANCESTOR.csv", sep='\t',low_memory=False) # from OHDSI

concept_df = concept_df[(concept_df['domain_id'] == 'Condition') &
                        (concept_df['vocabulary_id'] == 'SNOMED')]

concept_name_map = concept_df.set_index('concept_id')['concept_name'].to_dict()

merged = ancestor_df.merge(concept_df[['concept_id', 'concept_name']],
                           left_on='descendant_concept_id',
                           right_on='concept_id',
                           how='inner').rename(columns={'concept_name': 'descendant_name'})

merged = merged.merge(concept_df[['concept_id', 'concept_name']],
                      left_on='ancestor_concept_id',
                      right_on='concept_id',
                      how='left').rename(columns={'concept_name': 'ancestor_name'})

layered = merged[merged['min_levels_of_separation'] <= 100]

output = layered[['descendant_concept_id', 'descendant_name',
                  'ancestor_concept_id', 'ancestor_name',
                  'min_levels_of_separation']].drop_duplicates()

output.to_csv("concept_hierarchy_mapping.tsv", sep='\t', index=False)

In [ ]:
# For each disease of interest, count participants with any condition concept that descends from it,
# and plot the percentage of the 1,027 participants.
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

# Load files
hierarchy_df = pd.read_csv("concept_hierarchy_mapping.tsv", sep="\t")
condition_df = pd.read_csv("AoU_condition_with_details.csv")

total_samples = 1027

# Disease list
diseases_of_interest = [
    "Hypertensive disorder",
    "type 2 diabetes mellitus",
    "chronic kidney disease",
    "obesity",
    "Hypokalemia",
    "hyperglycemia",
    "hyperlipidemia",
    "glaucoma",
    "uterine leiomyoma",
    "osteoarthritis",
    "asthma",
    "obstructive sleep apnea",
    "anemia",
    "malignant neoplastic disease",
    "vitamin d deficiency",
    "heart disease",
    "bacterial infectious disease"
]
diseases_of_interest = [d.lower() for d in diseases_of_interest]

ancestor_to_descendants = defaultdict(set)
for _, row in hierarchy_df.iterrows():
    ancestor_name = str(row["ancestor_name"]).strip().lower()
    descendant_id = int(row["descendant_concept_id"])
    descendant_name = str(row["descendant_name"]).strip()
    ancestor_to_descendants[ancestor_name].add((descendant_id, descendant_name))
    
concept_to_person_ids = defaultdict(set)
for _, row in condition_df.iterrows():
    cid = int(row["concept_id"])
    pid = row["person_id"]
    concept_to_person_ids[cid].add(pid)
    
output_rows = []
plot_data = []

for disease in diseases_of_interest:
    related_concepts = ancestor_to_descendants.get(disease, set())
    if not related_concepts:
        continue
    concept_ids = {cid for cid, _ in related_concepts}
    concept_names = sorted({cname for _, cname in related_concepts})

    all_matched_pids = set()
    for cid in concept_ids:
        all_matched_pids.update(concept_to_person_ids.get(cid, set()))

    if not all_matched_pids:
        continue

    n_individuals = len(all_matched_pids)
    percentage = (n_individuals / total_samples) * 100
    related_names_str = ":,:".join(concept_names)

    output_rows.append(f"{disease}\t{n_individuals}\t{related_names_str}")
    plot_data.append((disease.capitalize(), percentage))
    

plot_df = pd.DataFrame(plot_data, columns=["Disease", "Percentage"])
plot_df = plot_df.sort_values(by="Percentage", ascending=False)

plt.figure(figsize=(6,5))
plt.barh(plot_df["Disease"], plot_df["Percentage"], color="black")
plt.xlabel("Percentage of participants (%)")

plt.tight_layout()
plt.savefig("Enriched_diseases.png", dpi=600)

In [ ]:
# The same bar plot, saved as PDF with larger fonts.
plot_df = pd.DataFrame(plot_data, columns=["Disease", "Percentage"])
plot_df = plot_df.sort_values(by="Percentage", ascending=False)

plt.figure(figsize=(6,5))
plt.barh(plot_df["Disease"], plot_df["Percentage"], color="black")
plt.xlabel("Percentage of participants (%)", fontsize=14)

plt.xticks(fontsize=12)

plt.tight_layout()
plt.savefig("Enriched_diseases.pdf")

In [ ]:
# Number of distinct OMOP concepts per participant, stacked by domain, sorted by total.
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("AoU_LR_samples_all_domains.csv")
counts = df.groupby(['person_id', 'source'])['concept_id'].nunique().unstack(fill_value=0)

counts['total'] = counts.sum(axis=1)
counts_sorted = counts.sort_values(by='total', ascending=False).drop(columns='total')

long_df = counts_sorted.reset_index().melt(id_vars='person_id', var_name='source', value_name='unique_concept_count')
long_df = long_df[long_df['unique_concept_count'] > 0]
long_df.to_csv("unique_concept_counts_per_sample.tsv", sep='\t', index=False)

domain_labels = {
    "condition": "Condition occurrence",
    "observation": "Observation",
    "drug_exposure": "Drug exposure",
    "measurement": "Measurement",
    "procedure": "Procedure"
}

counts_sorted_renamed = counts_sorted.rename(columns={k: v for k, v in domain_labels.items() if k in counts_sorted.columns})

ax = counts_sorted_renamed.plot(kind='bar', stacked=True, figsize=(6.5, 4), colormap='tab10')
plt.ylabel("Count")
plt.xlabel("Sample")
plt.xticks([], []) 
legend = plt.legend(title="OMOP Domain", frameon=False) 
plt.tight_layout()
                                             
plt.savefig("stacked_bar_unique_concepts_sorted.png", dpi=600, bbox_inches='tight')

In [ ]:
# Supplementary table: for each disorder concept, its ancestor concept IDs and the number of long-read
# participants with that concept.
import pandas as pd

condition_file = "AoU_condition_with_details.csv"
df_conditions = pd.read_csv(condition_file)
df_conditions = df_conditions[df_conditions["concept_class_id"] == "Disorder"]

hierarchy_file = "concept_hierarchy_mapping.tsv"
df_hierarchy = pd.read_csv(hierarchy_file, sep="\t")

df_hierarchy = df_hierarchy[
    (df_hierarchy["descendant_concept_id"] != df_hierarchy["ancestor_concept_id"]) &
    (df_hierarchy["ancestor_name"].notna()) &
    (df_hierarchy["ancestor_name"].str.strip() != "")
]

ancestor_map = df_hierarchy.groupby("descendant_concept_id")["ancestor_concept_id"] \
    .apply(lambda x: ",".join(map(str, sorted(set(x))))).reset_index()

df_conditions = df_conditions.rename(columns={"concept_id": "descendant_concept_id"})
df_merged = pd.merge(df_conditions, ancestor_map, on="descendant_concept_id", how="left")

df_output = df_merged[["person_id", "descendant_concept_id", "concept_name", "ancestor_concept_id"]]
df_output = df_output.rename(columns={
    "descendant_concept_id": "concept_id",
    "ancestor_concept_id": "ancestor_concept_ids"
})

df_output["ancestor_concept_ids"] = df_output["ancestor_concept_ids"].fillna("")

summary = df_output.groupby(["concept_id", "concept_name", "ancestor_concept_ids"])["person_id"] \
                   .nunique().reset_index(name="num_samples")
summary = summary.sort_values(by="num_samples", ascending=False)
summary.to_csv("disease_summary_suptable.csv", index=False)

In [ ]:
!head -n 10 disease_summary_suptable.csv

In [ ]:
# Dataset Builder queries: demographics, survey answers, and ZIP3 socioeconomic data for the long-read cohort.
import pandas
import os

# This query represents dataset "Long reads" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_45339929_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) )"""

dataset_45339929_person_df = pandas.read_gbq(
    dataset_45339929_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_45339929_person_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "Long reads" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_45339929_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr                               
                WHERE
                    concept_id IN (1586134)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_lr_whole_genome_variant = 1 ) )
        )"""

dataset_45339929_survey_df = pandas.read_gbq(
    dataset_45339929_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_45339929_survey_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "Long reads" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v7
dataset_45339929_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) ) 
        AND observation_source_concept_id = 1585250 
        AND observation.value_as_string NOT LIKE 'Res%'"""

dataset_45339929_zip_code_socioeconomic_df = pandas.read_gbq(
    dataset_45339929_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_45339929_zip_code_socioeconomic_df.head(5)

In [ ]:
# Age distribution by sex at birth for participants aged 20 and over, in 10-year bins (age is computed
# from year of birth relative to the current date).
import pandas as pd
import matplotlib.pyplot as plt

# Assume your dataframe is already loaded as dataset_45339929_person_df

# Compute age
today = pd.Timestamp.now(tz="UTC")
dataset_45339929_person_df["age"] = (
    today.year - pd.to_datetime(dataset_45339929_person_df["date_of_birth"]).dt.year
)

# Optional: Clean up sex_at_birth
df = dataset_45339929_person_df.copy()
df = df[df["sex_at_birth"].isin(["Male", "Female"])]
df = df[df["age"] >= 20]

# Define age bins
bins = range(20, 100, 10)
df["age_bin"] = pd.cut(df["age"], bins=bins)

# Group by age_bin and sex_at_birth
grouped = df.groupby(["age_bin", "sex_at_birth"]).size().unstack(fill_value=0)

# Plot
fig, ax = plt.subplots(figsize=(5, 5))  # 1:1 aspect ratio

grouped.plot(kind="bar", stacked=False, color=["#DF5A49", "#45B29D"], ax=ax)

# Increase font sizes
ax.set_xlabel("Age Range", fontsize=14)
ax.set_ylabel("Number of participants", fontsize=14)
ax.legend(title="Sex at Birth", fontsize=12, title_fontsize=12)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=12)

# Custom x-axis labels
#ax.set_xticklabels([f"{int(interval.left)}–{int(interval.right-1)}" for interval in grouped.index], rotation=45)
ax.set_xticklabels([f"{int(interval.left)}–{int(interval.right-1)}" for interval in grouped.index], rotation=0)

plt.tight_layout()

# Save plot
plt.savefig("Age_Distribution_by_Sex_20plus.pdf")

# Show plot
plt.show()